In [6]:
from kg_agent import graph as kg
from kg_agent.tools import TOOLS, repurposing_candidates

g = kg.load_graph()
print(g.number_of_nodes(), "nodes,", g.number_of_edges(), "edges")

print("Targets of drug niraparib: ", kg.targets_of_drug(g, "niraparib"))
print("Drugs for disease Asthma: ", kg.drugs_for_disease(g, "Asthma"))
print("Repurposing candidates Mepolizumab: ", kg.repurposing_candidates(g, "Mepolizumab"))
print("Connection path Umeclidinium and Salbutamol: ", kg.connection_path(g, "Umeclidinium", "Salbutamol"))

# What the LLM actually sees: tool name, description and argument schema
for t in TOOLS:
    print('='*80)
    # print(t)
    # print('='*80)
    print("tool name, description and argument schema")
    print(t.name, "|", t.description, "|", t.args)

# A tool call exactly as the agent would make it
print("Repurposing candidates for drug mepolizumab", repurposing_candidates.invoke({"drug": "mepolizumab"}))
print("Repurposing candidates for drug incorrect mepolizumabb", repurposing_candidates.invoke({"drug": "mepolizumabb"}))   # typo: note the "did you mean" hint

25 nodes, 22 edges
Targets of drug niraparib:  [('Niraparib', 'TARGETS', 'PARP1'), ('Niraparib', 'TARGETS', 'PARP2')]
Drugs for disease Asthma:  [('Fluticasone propionate', 'TREATS', 'Asthma'), ('Salbutamol', 'TREATS', 'Asthma'), ('Mepolizumab', 'TREATS', 'Severe eosinophilic asthma')]
Repurposing candidates Mepolizumab:  [{'disease': 'EGPA', 'evidence': [('Mepolizumab', 'TARGETS', 'IL5'), ('IL5', 'ASSOCIATED_WITH', 'EGPA')]}]
Connection path Umeclidinium and Salbutamol:  [('Umeclidinium', 'TREATS', 'COPD'), ('Salbutamol', 'TREATS', 'COPD')]
tool name, description and argument schema
targets_of_drug | Return molecular targets of a drug as (drug, TARGETS, target) triples. | {'drug': {'title': 'Drug', 'type': 'string'}}
tool name, description and argument schema
drugs_for_disease | Return drugs that treat a disease, including drugs for its subtypes. | {'disease': {'title': 'Disease', 'type': 'string'}}
tool name, description and argument schema
repurposing_candidates | Propose diseases l

In [1]:
import os, getpass
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("API key: ")   # typed, never saved in the notebook
os.environ["KG_MODEL"] = "anthropic:claude-haiku-5-5"

from kg_agent.agent import build_agent

agent = build_agent()
state = agent.invoke({"messages": [{"role": "user",
                                    "content": "How is mepolizumab connected to EGPA, and is that a treatment?"}]})

for m in state["messages"]:
    print(f"\n--- {m.type.upper()}")
    for tc in getattr(m, "tool_calls", None) or []:
        print(f"  calls {tc['name']}({tc['args']})")
    if m.text:
        print(" ", m.text[:600])


--- HUMAN
  How is mepolizumab connected to EGPA, and is that a treatment?

--- AI
  calls connection_path({'entity_a': 'mepolizumab', 'entity_b': 'EGPA'})
  calls drugs_for_disease({'disease': 'EGPA'})

--- TOOL
  {"result": [["Mepolizumab", "TARGETS", "IL5"], ["IL5", "ASSOCIATED_WITH", "EGPA"]]}

--- TOOL
  {"result": []}

--- AI
  Mepolizumab is connected to EGPA through its target, but the graph does not record it as a treatment for EGPA.

- **The connection:** The shortest path runs through interleukin-5. (Mepolizumab, TARGETS, IL5) and (IL5, ASSOCIATED_WITH, EGPA).
- **Treatment status:** A query for drugs that treat EGPA returned no results, so there is no (mepolizumab, TREATS, EGPA) triple in this graph.

So the graph links mepolizumab to EGPA only indirectly, via IL-5, which is the pathway the drug blocks. It does not establish that mepolizumab is used to treat EGPA. That gap may reflect incomplete coverage in thi
